### CEBRA EMG model trial

In [32]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from cebra import CEBRA

Functions 

In [14]:
def load_emg_npz(npz_path):
    data = np.load(npz_path)
    emg = data['data']  # <- changed from 'emg' to 'data'
    
    # Simulate labels based on dorsiflexion cycles
    n_samples = emg.shape[0]
    cycle_length = 2000  # samples per dorsiflexion cycle
    pattern = np.concatenate([
        np.full(400, 0),  # neutral
        np.full(400, 1),  # mid
        np.full(400, 2),  # max
        np.full(400, 1),  # mid
        np.full(400, 0)   # back to neutral
    ])
    n_cycles = n_samples // cycle_length
    labels = np.tile(pattern, n_cycles)[:n_samples]
    
    return emg, labels


In [15]:
def preprocess_emg(emg):
    scaler = StandardScaler()
    emg_normalized = scaler.fit_transform(emg)
    return emg_normalized

In [16]:
def segment_emg_for_cebra(emg, labels, window_size=100, step_size=50):
    """
    Segment EMG signal into overlapping windows for CEBRA.
    Each segment becomes one input example.
    """
    X = []
    y = []

    for start in range(0, len(emg) - window_size, step_size):
        end = start + window_size
        X.append(emg[start:end])
        
        # Majority label in the window
        label = np.bincount(labels[start:end]).argmax()
        y.append(label)

    return np.array(X), np.array(y)

In [17]:
def prepare_cebra_dataset(X, y):
    # Flatten each window: shape (n_samples, window_size * n_channels)
    X_flat = X.reshape(X.shape[0], -1)
    return X_flat, y

In [27]:
# Define a CEBRA model
cebra_model = CEBRA(
    model_architecture="offset10-model", # also 'offset10-model', 'offset10-model-mse', 'offset5-model', 'offset1-model-mse'
    batch_size=512,
    learning_rate=3e-4,
    temperature=1.12, # factor to scale the similarity of the pairs
    max_iterations=5000, # default
    conditional='time', # for unsupervised. for supervised 'time_delta', or 'delta'
    output_dimension=3,
    distance='cosine', # also 'euclidean'
    device="cuda_if_available",
    verbose=True,
    time_offsets=10 # distance (in time) between positive pairs
)

Model pipeline

In [28]:
# 1. Load and preprocess
emg, labels = load_emg_npz("dummy_data.npz")
emg_normalized = preprocess_emg(emg)
print(f"EMG shape: {emg.shape}")
print(f"Labels shape: {labels.shape}")

EMG shape: (1000000, 5)
Labels shape: (1000000,)


In [29]:
# 2. Segment and prepare
X, y = segment_emg_for_cebra(emg_normalized, labels)
X_cebra, y_cebra = prepare_cebra_dataset(X, y)
print(f"X window shape: {X.shape}")
print(f"y window shape: {y.shape}")
print(f"X_cebra shape: {X_cebra.shape}")
print(f"y_cebra shape: {y_cebra.shape}")

X window shape: (19998, 100, 5)
y window shape: (19998,)
X_cebra shape: (19998, 500)
y_cebra shape: (19998,)


In [30]:
# 3. Initialize CEBRA model

emg_model = cebra_model.fit(X_cebra, y_cebra)

pos: -0.8828 neg:  6.5994 total:  5.7166 temperature:  1.1200: 100%|██████████| 5000/5000 [05:49<00:00, 14.30it/s]


In [ ]:
# 4. Get low-dimensional embeddings
embeddings = cebra_model.transform(X_cebra)
print(f"Generated embeddings shape: {embeddings.shape}")

Generated embeddings shape: (19998, 3)


In [ ]:
# calculate GoF
gof_full = cebra.sklearn.metrics.goodness_of_fit_score(emg_model, labels)
print(" GoF in bits - full:", gof_full)

ValueError: Number of index invalid: labels must have the same number of index as for fitting,expects 1, got 0 idx.

In [ ]:
# plot embedding
fig = cebra.integrations.plotly.plot_embedding_interactive(cebra_time_full, embedding_labels=hippocampus_pos.continuous_index[:,0], title = "CEBRA-Time (full)", markersize=3, cmap = "rainbow")
fig.show()
# plot the loss curve
ax = cebra.plot_loss(cebra_time_full_model)